# Fashion-MNIST Image Classification

Trains a small fully-connected neural network on Fashion-MNIST: 70 000 grayscale 28×28 images spread across 10 clothing categories. Final model is saved to `models/fashion_mnist.keras` for the website Demo.

## Part 0 — Setup

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print('TensorFlow', tf.__version__)

## Part 1 — Load the Fashion-MNIST Dataset

Bundled with Keras: 60 000 train + 10 000 test grayscale images, 28×28.

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
print('Train:', train_images.shape, '  Test:', test_images.shape)

In [ ]:
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot',
]

### 1.1 Inspect the data

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_images[i], cmap='gray')
    ax.set_title(class_names[int(train_labels[i])], fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## Part 2 — Preprocess

Pixels are 0–255 ints; scale to floats in [0, 1] for stable training.

In [ ]:
train_images = train_images.astype('float32') / 255.0
test_images  = test_images.astype('float32') / 255.0

## Part 3 — Build the Model

Tiny, fast network: flatten → Dense(128) → Dense(10). Outputs softmax probabilities directly.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## Part 4 — Train

In [ ]:
EPOCHS = 10

history = model.fit(
    train_images, train_labels,
    epochs=EPOCHS,
    validation_data=(test_images, test_labels),
    verbose=1,
)

### 4.1 Plot training history

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['loss'], label='train')
ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('Epoch'); ax[0].legend()
ax[1].plot(history.history['accuracy'], label='train')
ax[1].plot(history.history['val_accuracy'], label='val')
ax[1].set_title('Accuracy'); ax[1].set_xlabel('Epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## Part 5 — Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=0)
print(f'Test loss     : {test_loss:.4f}')
print(f'Test accuracy : {test_acc:.4f}')

### 5.1 Visualise predictions on test samples

In [ ]:
preds = model.predict(test_images[:15], verbose=0)
fig, axes = plt.subplots(3, 5, figsize=(12, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i], cmap='gray')
    pred_idx = int(np.argmax(preds[i]))
    true_idx = int(test_labels[i])
    ok = pred_idx == true_idx
    title = f'{class_names[pred_idx]}\n(true: {class_names[true_idx]})'
    ax.set_title(title, color='green' if ok else 'red', fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

## Part 6 — Persist the Model

Saves to `models/fashion_mnist.keras` so the website backend can load it on demand.

In [ ]:
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / 'fashion_mnist.keras'
model.save(MODEL_PATH)
print('Saved ->', MODEL_PATH, f'({MODEL_PATH.stat().st_size/1024:.1f} KB)')

## Part 7 — Prediction Helper

Reusable function. Accepts file path / PIL image / numpy array, returns label + confidence.

In [ ]:
from PIL import Image, ImageOps

def predict_image(img_or_path):
    """Classify an image into one of the 10 Fashion-MNIST classes.

    Converts to grayscale, resizes to 28x28, inverts if needed (Fashion-MNIST is
    light items on dark bg).
    """
    if isinstance(img_or_path, (str, Path)):
        img = Image.open(img_or_path)
    elif isinstance(img_or_path, Image.Image):
        img = img_or_path
    else:
        img = Image.fromarray(np.asarray(img_or_path))

    img = img.convert('L').resize((28, 28))
    arr = np.asarray(img, dtype='float32')
    # If background looks bright, invert (Fashion-MNIST style is light item, dark bg)
    if arr.mean() > 127:
        arr = 255 - arr
    arr = arr / 255.0
    arr = np.expand_dims(arr, axis=0)
    probs = model.predict(arr, verbose=0)[0]
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx]), {c: float(p) for c, p in zip(class_names, probs)}

label, conf, _ = predict_image(test_images[0])
print(f'Test sample 0 -> {label} ({conf:.2%})')